# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/susheel123-sketch/Flyrank-Internship-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [4]:
# Build label: March vs Feb impressions change
agg_feb = perf_feb.groupby("content_hash_id").agg(gsc_impressions_feb=("gsc_impressions", "sum")).reset_index()

fv = agg_march.merge(agg_feb, on="content_hash_id", how="left")
fv["pct_change"] = (fv["gsc_impressions"] - fv["gsc_impressions_feb"]) / fv["gsc_impressions_feb"].replace(0, pd.NA)
fv["is_declining"] = (fv["pct_change"] < -0.10).astype(int)

fv = fv.merge(
    dim_content[["content_hash_id", "word_count", "content_type", "search_volume", "content_created_date"]],
    on="content_hash_id", how="left"
)

fv = pd.get_dummies(fv, columns=["content_type"], prefix="ctype")

for col in ["gsc_avg_position", "ga4_engaged_sessions", "scroll_events", "word_count", "search_volume"]:
    fv[col + "_was_missing"] = fv[col].isna().astype(int)
    fv[col] = fv[col].fillna(fv[col].median())

fv.head()

,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_engaged_sessions,scroll_events,gsc_impressions_feb,pct_change,is_declining,word_count,search_volume,content_created_date,ctype_comparison article,ctype_feedly article,ctype_keyword article,gsc_avg_position_was_missing,ga4_engaged_sessions_was_missing,scroll_events_was_missing,word_count_was_missing,search_volume_was_missing
0,content_000005d4ced12088,86,0,72.854861,0.0,0.0,24.0,2.583333,0,2516.0,110.0,2025-03-28,False,False,True,0,0,0,1,0
1,content_00001e488b74b799,0,0,8.505296,0.0,0.0,0.0,<NA>,0,2516.0,0.0,2025-04-18,False,False,True,1,0,0,1,0
2,content_00007bd2985b77c3,47,0,5.269565,0.0,0.0,16.0,1.9375,0,2516.0,10.0,2025-07-31,False,False,True,0,0,0,1,0
3,content_00008950670cb6b5,0,0,8.505296,1.0,1.0,0.0,<NA>,0,2005.0,110.0,2025-07-11,False,False,True,1,0,0,0,0
4,content_0000a348850eb1fc,0,0,8.505296,0.0,1.0,0.0,<NA>,0,811.0,10.0,2025-09-02,False,True,False,1,0,0,0,1


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

- **gsc_impressions, gsc_clicks, gsc_avg_position** (March aggregates): search visibility
  and ranking. Available before the decision moment — reflects past performance only.
- **ga4_engaged_sessions, scroll_events**: engagement signals. Missing for GSC-only
  clients, flagged with `_was_missing`. Available before the decision moment.
- **word_count, search_volume** (from `dim_content`): static page/keyword attributes.
  Missing values filled with median. Available before the decision moment.
- **content_type** (one-hot encoded): categorical page type. Available before the
  decision moment — it's a fixed attribute of the page.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

**What happened:** Adding `leaky_flag` (a direct copy of the label) pushed accuracy
to near-perfect. Removing it returned a realistic, honest score. A near-perfect result
is a red flag for leakage, not a sign of a strong model.

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

feature_cols = [c for c in fv.columns if c not in
    ["content_hash_id", "gsc_impressions_feb", "pct_change", "is_declining", "content_created_date"]]

X = fv[feature_cols].fillna(0)
y = fv["is_declining"]

# LEAKY: inject a column copied directly from the label
X_leaky = X.copy()
X_leaky["leaky_flag"] = y

X_train, X_test, y_train, y_test = train_test_split(X_leaky, y, test_size=0.25, random_state=42)
leaky_acc = accuracy_score(y_test, RandomForestClassifier(random_state=42).fit(X_train, y_train).predict(X_test))
print(f"LEAKY accuracy: {leaky_acc:.3f}")

# HONEST: same features, leak removed
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
honest_acc = accuracy_score(y_test, RandomForestClassifier(random_state=42).fit(X_train, y_train).predict(X_test))
print(f"HONEST accuracy: {honest_acc:.3f}")

LEAKY accuracy: 1.000
HONEST accuracy: 0.849


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- **gsc_impressions_feb, pct_change**: excluded from features — these are the inputs
  used to *construct* the label itself, so including them would leak the label back
  into the features.
- **last_optimized_date, optimization_eligible_date**: excluded — reflect whether a
  review action already happened, not a pre-decision signal.
- **leaky_flag**: excluded (added only to demonstrate leakage) — direct copy of the label.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.